## 1. Imports and settings

In [1]:
import os
import random
import json
import numpy as np
import pandas as pd
import optuna
import joblib

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


C:\Users\AbsoluteArm\anaconda3\envs\torch-gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
GPU: NVIDIA RTX 5000 Ada Generation Laptop GPU


## 2. Paths and user settings

In [3]:
# ============================
# Paths
# ============================

DATA_DIR = r"C:\Users\AbsoluteArm\OneDrive - University of Oklahoma (1)\Alloy Project\6coef_May25\Data"

OUTPUT_DIR = r"C:\Users\AbsoluteArm\OneDrive - University of Oklahoma (1)\Alloy Project\6coef_May25\pred"

os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_C_FILE = "dat_bin_odd_c.csv"
TRAIN_T_FILE = "dat_bin_odd_t.csv"

TEST_FILES = {
    "dat_test_bin_even_c.csv": 0.0,
    "dat_test_bin_even_t.csv": 1.0,
    "dat_test_qq_c.csv": 0.0,
    "dat_test_qq_t.csv": 1.0
}

# ============================
# Runtime controls
# ============================

# For quick testing, use 10–20.
# For final training, use 50–100.
N_TRIALS = {
    "m_log": 10,
    "m_bc": 10,
    "var_log": 10,
    "var_bc": 10
}

# Ensemble size.
# For quick testing: 3
# For final run: 5 or 10
N_ENSEMBLE = 3

# Cross-validation folds used inside Optuna
N_SPLITS = 5

# Training control
MAX_EPOCHS = 700
PATIENCE = 70


## 3. Column definitions

In [4]:
ELEMENT_COLS = ["Mo", "Nb", "Ta", "V", "W"]
TYPE_COL = "loading_type"

INPUT_COLS = ELEMENT_COLS + [TYPE_COL]

CONTEXT_COLS = ELEMENT_COLS + ["alloy_cnt", TYPE_COL]

M_LOG_COLS = ["m_log_max_strain"]

M_BC_COLS = [
    "m_bc_1", "m_bc_2", "m_bc_3",
    "m_bc_4", "m_bc_5", "m_bc_6"
]

VAR_LOG_COLS = ["var_log_max_strain"]

VAR_BC_COLS = [
    "var_bc_1", "var_bc_2", "var_bc_3",
    "var_bc_4", "var_bc_5", "var_bc_6"
]

OUTPUT_GROUPS = {
    "m_log": M_LOG_COLS,
    "m_bc": M_BC_COLS,
    "var_log": VAR_LOG_COLS,
    "var_bc": VAR_BC_COLS
}

OUTPUT_COLS = M_LOG_COLS + M_BC_COLS + VAR_LOG_COLS + VAR_BC_COLS

print("Model inputs:", INPUT_COLS)
print("Context columns:", CONTEXT_COLS)
print("Outputs:", OUTPUT_COLS)

Model inputs: ['Mo', 'Nb', 'Ta', 'V', 'W', 'loading_type']
Context columns: ['Mo', 'Nb', 'Ta', 'V', 'W', 'alloy_cnt', 'loading_type']
Outputs: ['m_log_max_strain', 'm_bc_1', 'm_bc_2', 'm_bc_3', 'm_bc_4', 'm_bc_5', 'm_bc_6', 'var_log_max_strain', 'var_bc_1', 'var_bc_2', 'var_bc_3', 'var_bc_4', 'var_bc_5', 'var_bc_6']


## 4. Data loading

In [5]:
def add_alloy_cnt_if_missing(df):
    """
    alloy_cnt is metadata only.
    If it is missing, create it as the number of nonzero composition columns.
    It is NOT used as a model input.
    """
    df = df.copy()
    if "alloy_cnt" not in df.columns:
        df["alloy_cnt"] = (df[ELEMENT_COLS].abs() > 1e-12).sum(axis=1)
    return df


def load_training_data():
    c_path = os.path.join(DATA_DIR, TRAIN_C_FILE)
    t_path = os.path.join(DATA_DIR, TRAIN_T_FILE)

    print("Reading:", c_path)
    print("Exists:", os.path.exists(c_path))
    print("Reading:", t_path)
    print("Exists:", os.path.exists(t_path))

    df_c = pd.read_csv(c_path)
    df_t = pd.read_csv(t_path)

    df_c[TYPE_COL] = 0
    df_t[TYPE_COL] = 1

    df = pd.concat([df_c, df_t], axis=0, ignore_index=True)
    df = df.replace([np.inf, -np.inf], np.nan)
    df = add_alloy_cnt_if_missing(df)

    # Training rows need complete inputs and outputs.
    # We do not remove outliers.
    df = df.dropna(subset=INPUT_COLS + OUTPUT_COLS).reset_index(drop=True)

    df[TYPE_COL] = df[TYPE_COL].astype(int)

    return df


def load_test_data(file_name, loading_type_value):
    path = os.path.join(DATA_DIR, file_name)
    print("Reading test:", path)
    print("Exists:", os.path.exists(path))

    df = pd.read_csv(path)
    df = df.replace([np.inf, -np.inf], np.nan)
    df = add_alloy_cnt_if_missing(df)

    df[TYPE_COL] = int(loading_type_value)


    df = df.dropna(subset=INPUT_COLS).reset_index(drop=True)
    df[TYPE_COL] = df[TYPE_COL].astype(int)

    return df


train_df = load_training_data()

print("\nTraining shape:", train_df.shape)
display(train_df.head())
display(train_df[OUTPUT_COLS].describe().T)


Reading: C:\Users\AbsoluteArm\OneDrive - University of Oklahoma (1)\Alloy Project\6coef_May25\Data\dat_bin_odd_c.csv
Exists: True
Reading: C:\Users\AbsoluteArm\OneDrive - University of Oklahoma (1)\Alloy Project\6coef_May25\Data\dat_bin_odd_t.csv
Exists: True

Training shape: (99, 21)


,Mo,Nb,Ta,V,W,alloy_cnt,m_log_max_strain,m_bc_1,m_bc_2,m_bc_3,...,m_bc_5,m_bc_6,var_log_max_strain,var_bc_1,var_bc_2,var_bc_3,var_bc_4,var_bc_5,var_bc_6,loading_type
0,0.9,0.1,0.0,0.0,0.0,3,0.060989,1.055743,0.934940,-1.002988,...,0.830942,-1.696834,-7.222051,-8.278252,-8.641709,-8.686253,-8.826926,-5.131413,-0.686159,0
1,0.9,0.0,0.1,0.0,0.0,4,-0.078072,1.032145,0.928169,-0.848310,...,0.700510,-0.464761,-6.491835,-7.463049,-7.800702,-8.121579,-7.778962,-5.873959,-1.429050,0
2,0.9,0.0,0.0,0.1,0.0,5,0.297954,1.174112,1.069752,-0.943338,...,1.159345,-1.172475,-5.252851,-6.194926,-6.440312,-7.319466,-6.539760,-5.144256,-0.068493,0
3,0.9,0.0,0.0,0.0,0.1,6,0.430205,1.679382,1.516917,-1.190123,...,1.997190,-0.932214,-9.205010,-9.920465,-10.130285,-12.246138,-8.537368,-7.606827,-6.559912,0
4,0.7,0.3,0.0,0.0,0.0,13,-0.429983,0.316160,0.180642,-0.604434,...,0.185432,0.027580,-5.553295,-7.001461,-7.260515,-7.627838,-8.154489,-5.617833,-4.057368,0


,count,mean,std,min,25%,50%,75%,max
m_log_max_strain,99.0,-0.008197,0.996600,-2.175087,-0.634312,-0.078072,0.440986,2.539137
m_bc_1,99.0,0.001050,0.999944,-2.016120,-0.598855,0.008424,0.708689,2.085089
m_bc_2,99.0,-0.000658,0.999978,-1.938234,-0.745690,0.007476,0.818448,2.042097
m_bc_3,99.0,-0.008464,0.996375,-4.181971,-0.549014,-0.131340,0.404278,3.652194
m_bc_4,99.0,-0.005640,0.998392,-2.671583,-0.596867,-0.180861,0.478123,3.963556
m_bc_5,99.0,0.007696,0.997004,-2.113162,-0.649827,-0.086471,0.637061,4.579897
m_bc_6,99.0,-0.000260,0.999997,-2.682026,-0.436632,0.110316,0.383629,3.742862
var_log_max_strain,99.0,-8.208935,11.229189,-62.739136,-7.202249,-6.046082,-4.809660,-2.507955
var_bc_1,99.0,-7.066471,1.815190,-13.561809,-7.461785,-6.671620,-5.865981,-4.236230
var_bc_2,99.0,-7.455628,1.921616,-15.804852,-7.998767,-7.212604,-6.139486,-3.668636


## 5. Convert training data to arrays

In [6]:
X_elements_all = train_df[ELEMENT_COLS].values.astype(np.float32)
X_load_all = train_df[TYPE_COL].values.astype(np.int64)

y_all_groups = {
    group_name: train_df[cols].values.astype(np.float32)
    for group_name, cols in OUTPUT_GROUPS.items()
}

print("X_elements:", X_elements_all.shape)
print("X_load:", X_load_all.shape)
for k, v in y_all_groups.items():
    print(k, v.shape)


X_elements: (99, 5)
X_load: (99,)
m_log (99, 1)
m_bc (99, 6)
var_log (99, 1)
var_bc (99, 6)


## 6. Dataset class

In [7]:
class AlloyGroupDataset(Dataset):
    def __init__(self, X_elements, X_load, y):
        self.X_elements = torch.tensor(X_elements, dtype=torch.float32)
        self.X_load = torch.tensor(X_load, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X_elements)

    def __getitem__(self, idx):
        return self.X_elements[idx], self.X_load[idx], self.y[idx]


## 7. Model architecture: Element-Embedding + Mixture-of-Experts

In [8]:
def get_activation(name):
    if name == "ReLU":
        return nn.ReLU
    if name == "LeakyReLU":
        return nn.LeakyReLU
    if name == "ELU":
        return nn.ELU
    if name == "Tanh":
        return nn.Tanh
    raise ValueError(f"Unknown activation: {name}")


def make_mlp(input_dim, hidden_dim, output_dim, n_layers, activation_name, dropout):
    """
    Builds an MLP with n_layers hidden layers.
    If n_layers = 0, it returns a direct linear layer.
    """
    activation = get_activation(activation_name)

    layers = []
    prev_dim = input_dim

    for _ in range(n_layers):
        layers.append(nn.Linear(prev_dim, hidden_dim))
        layers.append(activation())
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        prev_dim = hidden_dim

    layers.append(nn.Linear(prev_dim, output_dim))
    return nn.Sequential(*layers)


class ElementMoERegressor(nn.Module):
    """
    Element-embedding Mixture-of-Experts model.

    Inputs:
        X_elements: [batch, 5] composition fractions for Mo, Nb, Ta, V, W
        X_load:     [batch] loading type, 0 = compression, 1 = tension

    Output:
        y_pred: [batch, output_dim]
    """
    def __init__(
        self,
        output_dim,
        emb_dim,
        elem_hidden,
        elem_layers,
        load_emb_dim,
        n_experts,
        expert_hidden,
        expert_layers,
        latent_dim,
        head_hidden,
        head_layers,
        activation_name,
        dropout
    ):
        super().__init__()

        self.output_dim = output_dim
        self.n_elements = len(ELEMENT_COLS)

        # Element identity embedding: Mo, Nb, Ta, V, W
        self.element_embedding = nn.Embedding(self.n_elements, emb_dim)

        # Each element receives [element_embedding, composition_fraction]
        self.element_encoder = make_mlp(
            input_dim=emb_dim + 1,
            hidden_dim=elem_hidden,
            output_dim=latent_dim,
            n_layers=elem_layers,
            activation_name=activation_name,
            dropout=dropout
        )

        # Loading type embedding: compression/tension
        self.loading_embedding = nn.Embedding(2, load_emb_dim)

        base_dim = latent_dim + load_emb_dim

        # Gating network chooses the expert weights
        self.gate = nn.Linear(base_dim, n_experts)

        # Expert subnetworks
        self.experts = nn.ModuleList([
            make_mlp(
                input_dim=base_dim,
                hidden_dim=expert_hidden,
                output_dim=latent_dim,
                n_layers=expert_layers,
                activation_name=activation_name,
                dropout=dropout
            )
            for _ in range(n_experts)
        ])

        # Final prediction head
        self.head = make_mlp(
            input_dim=latent_dim,
            hidden_dim=head_hidden,
            output_dim=output_dim,
            n_layers=head_layers,
            activation_name=activation_name,
            dropout=dropout
        )

    def forward(self, X_elements, X_load):
        batch_size = X_elements.shape[0]

        element_ids = torch.arange(
            self.n_elements,
            device=X_elements.device,
            dtype=torch.long
        )

        elem_emb = self.element_embedding(element_ids)  # [5, emb_dim]
        elem_emb = elem_emb.unsqueeze(0).expand(batch_size, -1, -1)  # [B, 5, emb_dim]

        fractions = X_elements.unsqueeze(-1)  # [B, 5, 1]

        elem_input = torch.cat([elem_emb, fractions], dim=-1)  # [B, 5, emb_dim+1]

        elem_encoded = self.element_encoder(elem_input)  # [B, 5, latent_dim]

        # Weighted/summed alloy representation.
        # Fractions are used again so elements with zero fraction contribute little.
        alloy_repr = (elem_encoded * fractions).sum(dim=1)  # [B, latent_dim]

        load_repr = self.loading_embedding(X_load)  # [B, load_emb_dim]

        base_repr = torch.cat([alloy_repr, load_repr], dim=1)

        gate_logits = self.gate(base_repr)
        gate_weights = torch.softmax(gate_logits, dim=1)  # [B, n_experts]

        expert_outputs = torch.stack(
            [expert(base_repr) for expert in self.experts],
            dim=1
        )  # [B, n_experts, latent_dim]

        moe_repr = (expert_outputs * gate_weights.unsqueeze(-1)).sum(dim=1)

        out = self.head(moe_repr)

        return out


## 8. Scaling, residual learning, and loss helpers

In [9]:
def create_scaler(name):
    if name == "StandardScaler":
        return StandardScaler()
    if name == "RobustScaler":
        return RobustScaler()
    raise ValueError(f"Unknown scaler: {name}")


def fit_target_transform(y_train, scaler_name, residual_learning):
    """
    If residual_learning=True:
        baseline = training mean in original target space.
        model learns y - baseline.
    If residual_learning=False:
        model learns y directly.

    Then the chosen scaler is fitted to the target being learned.
    """
    if residual_learning:
        baseline = y_train.mean(axis=0, keepdims=True)
        y_to_fit = y_train - baseline
    else:
        baseline = np.zeros((1, y_train.shape[1]), dtype=np.float32)
        y_to_fit = y_train

    scaler = create_scaler(scaler_name)
    y_scaled = scaler.fit_transform(y_to_fit)

    return {
        "scaler": scaler,
        "baseline": baseline.astype(np.float32),
        "residual_learning": residual_learning,
        "scaler_name": scaler_name
    }, y_scaled.astype(np.float32)


def transform_target(y, artifacts):
    if artifacts["residual_learning"]:
        y_to_transform = y - artifacts["baseline"]
    else:
        y_to_transform = y

    return artifacts["scaler"].transform(y_to_transform).astype(np.float32)


def inverse_transform_prediction(y_scaled_pred, artifacts):
    y_part = artifacts["scaler"].inverse_transform(y_scaled_pred)

    if artifacts["residual_learning"]:
        y_part = y_part + artifacts["baseline"]

    return y_part.astype(np.float32)


def get_loss_function(loss_name, huber_delta=1.0):
    if loss_name == "MSE":
        return nn.MSELoss()
    if loss_name == "SmoothL1":
        return nn.SmoothL1Loss()
    if loss_name == "Huber":
        return nn.HuberLoss(delta=huber_delta)
    raise ValueError(f"Unknown loss: {loss_name}")


## 9. Training and prediction functions

In [10]:
def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    max_epochs=700,
    patience=70
):
    best_val_loss = np.inf
    best_state = None
    patience_counter = 0
    best_epoch = 0

    for epoch in range(max_epochs):
        model.train()

        for X_elements, X_load, y in train_loader:
            X_elements = X_elements.to(device)
            X_load = X_load.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            pred = model(X_elements, X_load)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()

        model.eval()
        val_losses = []

        with torch.no_grad():
            for X_elements, X_load, y in val_loader:
                X_elements = X_elements.to(device)
                X_load = X_load.to(device)
                y = y.to(device)

                pred = model(X_elements, X_load)
                val_loss = criterion(pred, y)
                val_losses.append(val_loss.item())

        mean_val_loss = float(np.mean(val_losses))

        if mean_val_loss < best_val_loss:
            best_val_loss = mean_val_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            patience_counter = 0
            best_epoch = epoch + 1
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    model.load_state_dict(best_state)

    return best_val_loss, best_epoch


def train_full_model_for_epochs(
    model,
    train_loader,
    optimizer,
    criterion,
    n_epochs
):
    model.train()

    for epoch in range(n_epochs):
        losses = []

        for X_elements, X_load, y in train_loader:
            X_elements = X_elements.to(device)
            X_load = X_load.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            pred = model(X_elements, X_load)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()

            losses.append(loss.item())

        if epoch == 0 or (epoch + 1) % 50 == 0:
            print(f"Epoch {epoch+1}/{n_epochs} | Loss: {np.mean(losses):.6f}")


def predict_single_model(model, artifacts, X_elements, X_load, batch_size=256):
    ds = AlloyGroupDataset(
        X_elements,
        X_load,
        np.zeros((len(X_elements), model.output_dim), dtype=np.float32)
    )

    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)

    preds_scaled = []

    model.eval()

    with torch.no_grad():
        for Xb, Lb, _ in loader:
            Xb = Xb.to(device)
            Lb = Lb.to(device)
            pred = model(Xb, Lb).cpu().numpy()
            preds_scaled.append(pred)

    preds_scaled = np.vstack(preds_scaled)

    preds = inverse_transform_prediction(preds_scaled, artifacts)

    return preds


def predict_ensemble(model_bundle, X_elements, X_load):
    preds = []

    for item in model_bundle["members"]:
        model = item["model"]
        artifacts = item["artifacts"]
        pred = predict_single_model(model, artifacts, X_elements, X_load)
        preds.append(pred)

    return np.mean(preds, axis=0)


## 10. Optuna objective

In [11]:
def suggest_params(trial, output_dim):
    params = {
        "emb_dim": trial.suggest_categorical("emb_dim", [2, 4, 8, 16]),
        "elem_hidden": trial.suggest_categorical("elem_hidden", [8, 16, 32, 64]),
        "elem_layers": trial.suggest_int("elem_layers", 1, 5),
        "load_emb_dim": trial.suggest_categorical("load_emb_dim", [2, 4, 8, 16, 32]),
        "n_experts": trial.suggest_int("n_experts", 2, 4),
        "expert_hidden": trial.suggest_categorical("expert_hidden", [8, 16, 32, 64, 128]),
        "expert_layers": trial.suggest_int("expert_layers", 1, 5),
        "latent_dim": trial.suggest_categorical("latent_dim", [8, 16, 32, 64]),
        "head_hidden": trial.suggest_categorical("head_hidden", [8, 16, 32, 64, 128]),
        "head_layers": trial.suggest_int("head_layers", 0, 2),
        "activation_name": trial.suggest_categorical(
            "activation_name",
            ["ReLU", "LeakyReLU", "ELU", "Tanh"]
        ),
        "dropout": trial.suggest_float("dropout", 0.0, 0.30),
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-8, 1e-2, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [4, 8, 16, 32]),
        "optimizer": trial.suggest_categorical("optimizer", ["Adam", "AdamW"]),
        "loss": trial.suggest_categorical("loss", ["MSE", "SmoothL1", "Huber"]),
        "target_scaler": trial.suggest_categorical("target_scaler", ["StandardScaler", "RobustScaler"]),
        "residual_learning": trial.suggest_categorical("residual_learning", [False, True]),
    }

    if params["loss"] == "Huber":
        params["huber_delta"] = trial.suggest_float("huber_delta", 0.25, 3.0)
    else:
        params["huber_delta"] = 1.0

    return params


def build_model_from_params(params, output_dim):
    model = ElementMoERegressor(
        output_dim=output_dim,
        emb_dim=params["emb_dim"],
        elem_hidden=params["elem_hidden"],
        elem_layers=params["elem_layers"],
        load_emb_dim=params["load_emb_dim"],
        n_experts=params["n_experts"],
        expert_hidden=params["expert_hidden"],
        expert_layers=params["expert_layers"],
        latent_dim=params["latent_dim"],
        head_hidden=params["head_hidden"],
        head_layers=params["head_layers"],
        activation_name=params["activation_name"],
        dropout=params["dropout"]
    ).to(device)

    return model


def build_optimizer(params, model):
    if params["optimizer"] == "Adam":
        return torch.optim.Adam(
            model.parameters(),
            lr=params["learning_rate"],
            weight_decay=params["weight_decay"]
        )

    if params["optimizer"] == "AdamW":
        return torch.optim.AdamW(
            model.parameters(),
            lr=params["learning_rate"],
            weight_decay=params["weight_decay"]
        )

    raise ValueError(f"Unknown optimizer: {params['optimizer']}")


def objective_for_group(trial, group_name):
    set_seed(SEED)

    y_all = y_all_groups[group_name]
    output_dim = y_all.shape[1]

    params = suggest_params(trial, output_dim)

    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    fold_losses = []
    fold_epochs = []

    for train_idx, val_idx in kf.split(X_elements_all):
        X_train = X_elements_all[train_idx]
        X_val = X_elements_all[val_idx]

        L_train = X_load_all[train_idx]
        L_val = X_load_all[val_idx]

        y_train = y_all[train_idx]
        y_val = y_all[val_idx]

        artifacts, y_train_scaled = fit_target_transform(
            y_train,
            scaler_name=params["target_scaler"],
            residual_learning=params["residual_learning"]
        )

        y_val_scaled = transform_target(y_val, artifacts)

        train_ds = AlloyGroupDataset(X_train, L_train, y_train_scaled)
        val_ds = AlloyGroupDataset(X_val, L_val, y_val_scaled)

        train_loader = DataLoader(
            train_ds,
            batch_size=params["batch_size"],
            shuffle=True
        )

        val_loader = DataLoader(
            val_ds,
            batch_size=params["batch_size"],
            shuffle=False
        )

        model = build_model_from_params(params, output_dim)
        optimizer = build_optimizer(params, model)

        criterion = get_loss_function(
            params["loss"],
            huber_delta=params["huber_delta"]
        )

        val_loss, best_epoch = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            optimizer=optimizer,
            criterion=criterion,
            max_epochs=MAX_EPOCHS,
            patience=PATIENCE
        )

        fold_losses.append(val_loss)
        fold_epochs.append(best_epoch)

    trial.set_user_attr("mean_best_epoch", int(np.mean(fold_epochs)))

    return float(np.mean(fold_losses))


## 11. Run Optuna for all four networks

In [12]:
studies = {}
best_params_by_group = {}
best_epochs_by_group = {}

for group_name in OUTPUT_GROUPS.keys():
    print("\n" + "="*70)
    print(f"Tuning group: {group_name}")
    print("Outputs:", OUTPUT_GROUPS[group_name])
    print("="*70)

    study = optuna.create_study(direction="minimize")

    study.optimize(
        lambda trial, g=group_name: objective_for_group(trial, g),
        n_trials=N_TRIALS[group_name],
        show_progress_bar=True
    )

    studies[group_name] = study
    best_params_by_group[group_name] = study.best_params
    best_epochs_by_group[group_name] = study.best_trial.user_attrs["mean_best_epoch"]

    print(f"\nBest value for {group_name}:", study.best_value)
    print(f"Best epoch for {group_name}:", best_epochs_by_group[group_name])
    print("Best parameters:")
    for k, v in study.best_params.items():
        print(f"{k}: {v}")


[I 2026-05-25 14:58:55,421] A new study created in memory with name: no-name-a679f742-ded3-49f7-803a-1f4b149c72e7



Tuning group: m_log
Outputs: ['m_log_max_strain']


Best trial: 0. Best value: 0.100845:  10%|█         | 1/10 [04:14<38:10, 254.54s/it]

[I 2026-05-25 15:03:09,959] Trial 0 finished with value: 0.10084538396447895 and parameters: {'emb_dim': 16, 'elem_hidden': 8, 'elem_layers': 3, 'load_emb_dim': 2, 'n_experts': 4, 'expert_hidden': 8, 'expert_layers': 1, 'latent_dim': 32, 'head_hidden': 128, 'head_layers': 2, 'activation_name': 'ELU', 'dropout': 0.21739594375148472, 'learning_rate': 0.000237174814762891, 'weight_decay': 0.00155985149206437, 'batch_size': 4, 'optimizer': 'Adam', 'loss': 'Huber', 'target_scaler': 'RobustScaler', 'residual_learning': True, 'huber_delta': 1.3908766664534533}. Best is trial 0 with value: 0.10084538396447895.


Best trial: 1. Best value: 0.0626393:  20%|██        | 2/10 [07:22<28:45, 215.63s/it]

...

[I 2026-05-25 15:55:42,910] Trial 7 finished with value: 1.118596104780833 and parameters: {'emb_dim': 8, 'elem_hidden': 8, 'elem_layers': 2, 'load_emb_dim': 32, 'n_experts': 4, 'expert_hidden': 32, 'expert_layers': 4, 'latent_dim': 16, 'head_hidden': 64, 'head_layers': 0, 'activation_name': 'LeakyReLU', 'dropout': 0.1967350847576367, 'learning_rate': 1.1821543782652337e-05, 'weight_decay': 4.303042504692705e-08, 'batch_size': 8, 'optimizer': 'Adam', 'loss': 'MSE', 'target_scaler': 'StandardScaler', 'residual_learning': False}. Best is trial 0 with value: 0.357032610476017.


Best trial: 0. Best value: 0.357033:  90%|█████████ | 9/10 [11:52<01:39, 99.51s/it] 

[I 2026-05-25 15:57:08,017] Trial 8 finished with value: 1.0579769730567932 and parameters: {'emb_dim': 8, 'elem_hidden': 32, 'elem_layers': 5, 'load_emb_dim': 16, 'n_experts': 3, 'expert_hidden': 8, 'expert_layers': 4, 'latent_dim': 64, 'head_hidden': 32, 'head_layers': 2, 'activation_name': 'ELU', 'dropout': 0.014145246896227969, 'learning_rate': 1.1707069259592739e-05, 'weight_decay': 6.074325877234101e-06, 'batch_size': 32, 'optimizer': 'Adam', 'loss': 'MSE', 'target_scaler': 'StandardScaler', 'residual_learning': False}. Best is trial 0 with value: 0.357032610476017.


Best trial: 0. Best value: 0.357033: 100%|██████████| 10/10 [14:41<00:00, 88.18s/it] 

[I 2026-05-25 15:59:57,430] Trial 9 finished with value: 1.0674429476261138 and parameters: {'emb_dim': 2, 'elem_hidden': 64, 'elem_layers': 1, 'load_emb_dim': 2, 'n_experts': 2, 'expert_hidden': 16, 'expert_layers': 4, 'latent_dim': 16, 'head_hidden': 128, 'head_layers': 2, 'activation_name': 'LeakyReLU', 'dropout': 0.019414307065049107, 'learning_rate': 1.7021697043300275e-05, 'weight_decay': 0.0006795257342809453, 'batch_size': 4, 'optimizer': 'Adam', 'loss': 'MSE', 'target_scaler': 'StandardScaler', 'residual_learning': True}. Best is trial 0 with value: 0.357032610476017.

Best value for var_bc: 0.357032610476017
Best epoch for var_bc: 53
Best parameters:
emb_dim: 2
elem_hidden: 32
elem_layers: 5
load_emb_dim: 8
n_experts: 4
expert_hidden: 128
expert_layers: 4
latent_dim: 8
head_hidden: 64
head_layers: 1
activation_name: LeakyReLU
dropout: 0.2987440552859433
learning_rate: 0.001568948042183037
weight_decay: 3.8519684048606015e-08
batch_size: 16
optimizer: AdamW
loss: SmoothL1
targ

## 12. Train ensemble models using best Optuna parameters

In [13]:
def train_ensemble_for_group(group_name, n_ensemble=N_ENSEMBLE):
    output_cols = OUTPUT_GROUPS[group_name]
    y_all = y_all_groups[group_name]
    output_dim = y_all.shape[1]

    params = best_params_by_group[group_name].copy()
    best_epoch = best_epochs_by_group[group_name]

    print("\n" + "="*70)
    print(f"Training ensemble for group: {group_name}")
    print("Outputs:", output_cols)
    print("Ensemble size:", n_ensemble)
    print("Epochs per ensemble member:", best_epoch)
    print("="*70)

    members = []

    for member_idx in range(n_ensemble):
        seed = SEED + 1000 * member_idx + len(group_name)
        set_seed(seed)

        artifacts, y_scaled = fit_target_transform(
            y_all,
            scaler_name=params["target_scaler"],
            residual_learning=params["residual_learning"]
        )

        train_ds = AlloyGroupDataset(
            X_elements_all,
            X_load_all,
            y_scaled
        )

        train_loader = DataLoader(
            train_ds,
            batch_size=params["batch_size"],
            shuffle=True
        )

        model = build_model_from_params(params, output_dim)
        optimizer = build_optimizer(params, model)

        criterion = get_loss_function(
            params["loss"],
            huber_delta=params.get("huber_delta", 1.0)
        )

        print(f"\nTraining {group_name} ensemble member {member_idx+1}/{n_ensemble} | seed = {seed}")

        train_full_model_for_epochs(
            model=model,
            train_loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            n_epochs=best_epoch
        )

        members.append({
            "seed": seed,
            "model": model,
            "artifacts": artifacts
        })

    bundle = {
        "group_name": group_name,
        "output_cols": output_cols,
        "params": params,
        "best_epoch": best_epoch,
        "members": members
    }

    return bundle


model_bundles = {}

for group_name in OUTPUT_GROUPS.keys():
    model_bundles[group_name] = train_ensemble_for_group(
        group_name,
        n_ensemble=N_ENSEMBLE
    )

print("All ensemble models trained.")



Training ensemble for group: m_log
Outputs: ['m_log_max_strain']
Ensemble size: 3
Epochs per ensemble member: 157

Training m_log ensemble member 1/3 | seed = 47
Epoch 1/157 | Loss: 0.379624
Epoch 50/157 | Loss: 0.137515
Epoch 100/157 | Loss: 0.074231
Epoch 150/157 | Loss: 0.034039

Training m_log ensemble member 2/3 | seed = 1047
Epoch 1/157 | Loss: 0.388295
Epoch 50/157 | Loss: 0.067440
Epoch 100/157 | Loss: 0.071181
Epoch 150/157 | Loss: 0.042123

Training m_log ensemble member 3/3 | seed = 2047
Epoch 1/157 | Loss: 0.467104
Epoch 50/157 | Loss: 0.129701
Epoch 100/157 | Loss: 0.060190
Epoch 150/157 | Loss: 0.072168

Training ensemble for group: m_bc
Outputs: ['m_bc_1', 'm_bc_2', 'm_bc_3', 'm_bc_4', 'm_bc_5', 'm_bc_6']
Ensemble size: 3
Epochs per ensemble member: 237

Training m_bc ensemble member 1/3 | seed = 46
Epoch 1/237 | Loss: 0.407767
Epoch 50/237 | Loss: 0.141841
Epoch 100/237 | Loss: 0.119764
Epoch 150/237 | Loss: 0.085700
Epoch 200/237 | Loss: 0.098287

Training m_bc ensemb

## 13. Predict and save test files

In [14]:
def evaluate_metrics_if_available(df_test, y_pred, file_name):
    """
    Metrics are calculated only on rows where all true outputs are available.
    Prediction is still performed for all rows with valid inputs.
    """
    if not all(col in df_test.columns for col in OUTPUT_COLS):
        print("Test file does not contain all true output columns. Metrics skipped.")
        return None

    valid_mask = df_test[OUTPUT_COLS].notna().all(axis=1)
    n_metric = int(valid_mask.sum())

    if n_metric == 0:
        print("No rows with complete true outputs. Metrics skipped.")
        return None

    y_true_metric = df_test.loc[valid_mask, OUTPUT_COLS].values.astype(np.float32)
    y_pred_metric = y_pred[valid_mask.values]

    print("Rows used for metrics:", n_metric)
    print("Overall MAE:", mean_absolute_error(y_true_metric, y_pred_metric))
    print("Overall RMSE:", np.sqrt(mean_squared_error(y_true_metric, y_pred_metric)))

    rows = []

    for i, col in enumerate(OUTPUT_COLS):
        y_true_i = y_true_metric[:, i]
        y_pred_i = y_pred_metric[:, i]

        mae_i = mean_absolute_error(y_true_i, y_pred_i)
        rmse_i = np.sqrt(mean_squared_error(y_true_i, y_pred_i))

        try:
            r2_i = r2_score(y_true_i, y_pred_i)
        except Exception:
            r2_i = np.nan

        rows.append({
            "test_file": file_name,
            "output": col,
            "MAE": mae_i,
            "RMSE": rmse_i,
            "R2": r2_i,
            "rows_used_for_metric": n_metric,
            "rows_predicted": len(df_test)
        })

    metrics_df = pd.DataFrame(rows)
    display(metrics_df)

    return metrics_df


def predict_all_groups(df_test):
    X_elements = df_test[ELEMENT_COLS].values.astype(np.float32)
    X_load = df_test[TYPE_COL].values.astype(np.int64)

    pred_parts = []

    for group_name in ["m_log", "m_bc", "var_log", "var_bc"]:
        pred_group = predict_ensemble(
            model_bundles[group_name],
            X_elements,
            X_load
        )
        pred_parts.append(pred_group)

    y_pred = np.concatenate(pred_parts, axis=1)

    return y_pred


def evaluate_and_save_test(file_name, loading_type_value):
    df_test = load_test_data(file_name, loading_type_value)

    if len(df_test) == 0:
        print(f"No valid input rows for {file_name}")
        return None, None

    y_pred = predict_all_groups(df_test)

    pred_output_cols = [f"pred_{col}" for col in OUTPUT_COLS]

    save_cols = [
        col for col in CONTEXT_COLS
        if col in df_test.columns
    ]

    pred_df = df_test[save_cols].copy()

    for j, col in enumerate(pred_output_cols):
        pred_df[col] = y_pred[:, j]

    save_name = file_name.replace(
        ".csv",
        "_predictions_4net_residual_ensemble_nn.csv"
    )

    save_path = os.path.join(OUTPUT_DIR, save_name)

    pred_df.to_csv(save_path, index=False)

    print("\n==============================")
    print("Test file:", file_name)
    print("Rows predicted:", len(pred_df))
    print("Predictions saved to:", save_path)

    metrics_df = evaluate_metrics_if_available(df_test, y_pred, file_name)

    return metrics_df, pred_df


all_metrics = []
all_predictions = {}

for file_name, loading_type_value in TEST_FILES.items():
    metrics_df, pred_df = evaluate_and_save_test(file_name, loading_type_value)
    all_predictions[file_name] = pred_df

    if metrics_df is not None:
        all_metrics.append(metrics_df)

if len(all_metrics) > 0:
    all_metrics_df = pd.concat(all_metrics, axis=0, ignore_index=True)

    metrics_path = os.path.join(
        OUTPUT_DIR,
        "test_metrics_4net_residual_ensemble_nn.csv"
    )

    all_metrics_df.to_csv(metrics_path, index=False)

    print("\nSaved metrics to:", metrics_path)
    display(all_metrics_df)

else:
    print("No metrics were generated.")


Reading test: C:\Users\AbsoluteArm\OneDrive - University of Oklahoma (1)\Alloy Project\6coef_May25\Data\dat_test_bin_even_c.csv
Exists: True

Test file: dat_test_bin_even_c.csv
Rows predicted: 40
Predictions saved to: C:\Users\AbsoluteArm\OneDrive - University of Oklahoma (1)\Alloy Project\6coef_May25\pred\dat_test_bin_even_c_predictions_4net_residual_ensemble_nn.csv
Rows used for metrics: 39
Overall MAE: 0.9704793691635132
Overall RMSE: 2.8069899095463313


,test_file,output,MAE,RMSE,R2,rows_used_for_metric,rows_predicted
0,dat_test_bin_even_c.csv,m_log_max_strain,0.210039,0.257669,0.929290,39,40
1,dat_test_bin_even_c.csv,m_bc_1,0.110251,0.149003,0.975375,39,40
2,dat_test_bin_even_c.csv,m_bc_2,0.112626,0.146765,0.975360,39,40
3,dat_test_bin_even_c.csv,m_bc_3,0.147327,0.185906,0.946883,39,40
4,dat_test_bin_even_c.csv,m_bc_4,0.155182,0.221948,0.936007,39,40
5,dat_test_bin_even_c.csv,m_bc_5,0.191022,0.269788,0.916574,39,40
6,dat_test_bin_even_c.csv,m_bc_6,0.441888,0.567316,0.676922,39,40
7,dat_test_bin_even_c.csv,var_log_max_strain,2.744193,9.214433,-0.037671,39,40
8,dat_test_bin_even_c.csv,var_bc_1,1.632429,2.215656,-0.222792,39,40
9,dat_test_bin_even_c.csv,var_bc_2,1.555039,2.058340,-0.097617,39,40


Reading test: C:\Users\AbsoluteArm\OneDrive - University of Oklahoma (1)\Alloy Project\6coef_May25\Data\dat_test_bin_even_t.csv
Exists: True

Test file: dat_test_bin_even_t.csv
Rows predicted: 40
Predictions saved to: C:\Users\AbsoluteArm\OneDrive - University of Oklahoma (1)\Alloy Project\6coef_May25\pred\dat_test_bin_even_t_predictions_4net_residual_ensemble_nn.csv
Rows used for metrics: 39
Overall MAE: 0.9127979278564453
Overall RMSE: 2.689750527509575


,test_file,output,MAE,RMSE,R2,rows_used_for_metric,rows_predicted
0,dat_test_bin_even_t.csv,m_log_max_strain,0.118638,0.146072,0.976336,39,40
1,dat_test_bin_even_t.csv,m_bc_1,0.106797,0.131017,0.982372,39,40
2,dat_test_bin_even_t.csv,m_bc_2,0.118590,0.149653,0.976181,39,40
3,dat_test_bin_even_t.csv,m_bc_3,0.199663,0.361461,0.875195,39,40
4,dat_test_bin_even_t.csv,m_bc_4,0.209445,0.371900,0.866816,39,40
5,dat_test_bin_even_t.csv,m_bc_5,0.323490,0.544625,0.698798,39,40
6,dat_test_bin_even_t.csv,m_bc_6,0.369784,0.539535,0.305891,39,40
7,dat_test_bin_even_t.csv,var_log_max_strain,2.691492,8.855338,0.009818,39,40
8,dat_test_bin_even_t.csv,var_bc_1,1.198973,1.728455,-0.001686,39,40
9,dat_test_bin_even_t.csv,var_bc_2,1.150728,1.559969,-0.000361,39,40


Reading test: C:\Users\AbsoluteArm\OneDrive - University of Oklahoma (1)\Alloy Project\6coef_May25\Data\dat_test_qq_c.csv
Exists: True

Test file: dat_test_qq_c.csv
Rows predicted: 41
Predictions saved to: C:\Users\AbsoluteArm\OneDrive - University of Oklahoma (1)\Alloy Project\6coef_May25\pred\dat_test_qq_c_predictions_4net_residual_ensemble_nn.csv
Rows used for metrics: 25
Overall MAE: 1.1967202425003052
Overall RMSE: 2.325629003614113


,test_file,output,MAE,RMSE,R2,rows_used_for_metric,rows_predicted
0,dat_test_qq_c.csv,m_log_max_strain,0.289554,0.340511,-0.836140,25,41
1,dat_test_qq_c.csv,m_bc_1,0.195772,0.237195,0.765436,25,41
2,dat_test_qq_c.csv,m_bc_2,0.176949,0.219482,0.785173,25,41
3,dat_test_qq_c.csv,m_bc_3,0.221411,0.244770,-0.044517,25,41
4,dat_test_qq_c.csv,m_bc_4,0.288024,0.408604,-0.318753,25,41
5,dat_test_qq_c.csv,m_bc_5,1.006555,1.397135,-0.881108,25,41
6,dat_test_qq_c.csv,m_bc_6,4.834455,6.932637,-0.893862,25,41
7,dat_test_qq_c.csv,var_log_max_strain,0.980576,1.344331,-0.060399,25,41
8,dat_test_qq_c.csv,var_bc_1,1.176231,1.458708,-0.109559,25,41
9,dat_test_qq_c.csv,var_bc_2,1.137656,1.404107,-0.051912,25,41


Reading test: C:\Users\AbsoluteArm\OneDrive - University of Oklahoma (1)\Alloy Project\6coef_May25\Data\dat_test_qq_t.csv
Exists: True

Test file: dat_test_qq_t.csv
Rows predicted: 41
Predictions saved to: C:\Users\AbsoluteArm\OneDrive - University of Oklahoma (1)\Alloy Project\6coef_May25\pred\dat_test_qq_t_predictions_4net_residual_ensemble_nn.csv
Rows used for metrics: 25
Overall MAE: 1.8328386545181274
Overall RMSE: 5.561078693669297


,test_file,output,MAE,RMSE,R2,rows_used_for_metric,rows_predicted
0,dat_test_qq_t.csv,m_log_max_strain,0.479112,0.596891,-0.624071,25,41
1,dat_test_qq_t.csv,m_bc_1,0.231541,0.292805,0.590458,25,41
2,dat_test_qq_t.csv,m_bc_2,0.241496,0.314844,0.686321,25,41
3,dat_test_qq_t.csv,m_bc_3,0.739550,0.939494,-0.480579,25,41
4,dat_test_qq_t.csv,m_bc_4,0.740854,1.344100,0.109024,25,41
5,dat_test_qq_t.csv,m_bc_5,1.792328,3.533351,-0.082224,25,41
6,dat_test_qq_t.csv,m_bc_6,2.127067,3.910225,0.059604,25,41
7,dat_test_qq_t.csv,var_log_max_strain,7.909117,19.136488,-0.088885,25,41
8,dat_test_qq_t.csv,var_bc_1,1.627437,2.094194,-0.039711,25,41
9,dat_test_qq_t.csv,var_bc_2,1.750159,2.438561,-0.004786,25,41



Saved metrics to: C:\Users\AbsoluteArm\OneDrive - University of Oklahoma (1)\Alloy Project\6coef_May25\pred\test_metrics_4net_residual_ensemble_nn.csv


,test_file,output,MAE,RMSE,R2,rows_used_for_metric,rows_predicted
0,dat_test_bin_even_c.csv,m_log_max_strain,0.210039,0.257669,0.929290,39,40
1,dat_test_bin_even_c.csv,m_bc_1,0.110251,0.149003,0.975375,39,40
2,dat_test_bin_even_c.csv,m_bc_2,0.112626,0.146765,0.975360,39,40
3,dat_test_bin_even_c.csv,m_bc_3,0.147327,0.185906,0.946883,39,40
4,dat_test_bin_even_c.csv,m_bc_4,0.155182,0.221948,0.936007,39,40
5,dat_test_bin_even_c.csv,m_bc_5,0.191022,0.269788,0.916574,39,40
6,dat_test_bin_even_c.csv,m_bc_6,0.441888,0.567316,0.676922,39,40
7,dat_test_bin_even_c.csv,var_log_max_strain,2.744193,9.214433,-0.037671,39,40
8,dat_test_bin_even_c.csv,var_bc_1,1.632429,2.215656,-0.222792,39,40
9,dat_test_bin_even_c.csv,var_bc_2,1.555039,2.058340,-0.097617,39,40


## 14. Save models, scalers, and Optuna summaries

In [15]:
# Save Optuna best parameters
params_path = os.path.join(OUTPUT_DIR, "best_params_4net_residual_ensemble_nn.json")
epochs_path = os.path.join(OUTPUT_DIR, "best_epochs_4net_residual_ensemble_nn.json")

with open(params_path, "w") as f:
    json.dump(best_params_by_group, f, indent=4)

with open(epochs_path, "w") as f:
    json.dump(best_epochs_by_group, f, indent=4)

print("Saved best params to:", params_path)
print("Saved best epochs to:", epochs_path)

# Save trained ensemble state dicts and preprocessing artifacts
# The model class definition is in this notebook, so these can be reloaded in the same code environment.

for group_name, bundle in model_bundles.items():
    group_dir = os.path.join(OUTPUT_DIR, f"saved_{group_name}")
    os.makedirs(group_dir, exist_ok=True)

    meta = {
        "group_name": bundle["group_name"],
        "output_cols": bundle["output_cols"],
        "params": bundle["params"],
        "best_epoch": bundle["best_epoch"],
        "n_ensemble": len(bundle["members"])
    }

    with open(os.path.join(group_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=4)

    for i, member in enumerate(bundle["members"]):
        torch.save(
            member["model"].state_dict(),
            os.path.join(group_dir, f"model_member_{i+1}.pt")
        )

        joblib.dump(
            member["artifacts"],
            os.path.join(group_dir, f"artifacts_member_{i+1}.pkl")
        )

print("Saved all model state dicts and scalers/artifacts.")


Saved best params to: C:\Users\AbsoluteArm\OneDrive - University of Oklahoma (1)\Alloy Project\6coef_May25\pred\best_params_4net_residual_ensemble_nn.json
Saved best epochs to: C:\Users\AbsoluteArm\OneDrive - University of Oklahoma (1)\Alloy Project\6coef_May25\pred\best_epochs_4net_residual_ensemble_nn.json
Saved all model state dicts and scalers/artifacts.


## 15. Optional diagnostic: prediction spread for variance outputs

In [16]:
def variance_spread_diagnostic_from_saved_predictions(file_name):
    pred_name = file_name.replace(
        ".csv",
        "_predictions_4net_residual_ensemble_nn.csv"
    )

    true_path = os.path.join(DATA_DIR, file_name)
    pred_path = os.path.join(OUTPUT_DIR, pred_name)

    df_true = pd.read_csv(true_path).replace([np.inf, -np.inf], np.nan)
    df_pred = pd.read_csv(pred_path).replace([np.inf, -np.inf], np.nan)

    rows = []

    for col in VAR_LOG_COLS + VAR_BC_COLS:
        pred_col = f"pred_{col}"

        if col not in df_true.columns or pred_col not in df_pred.columns:
            continue

        valid_mask = df_true[col].notna() & df_pred[pred_col].notna()

        if valid_mask.sum() == 0:
            continue

        y_true = df_true.loc[valid_mask, col].values
        y_pred = df_pred.loc[valid_mask, pred_col].values

        true_std = np.std(y_true)
        pred_std = np.std(y_pred)

        rows.append({
            "test_file": file_name,
            "output": col,
            "n": int(valid_mask.sum()),
            "true_mean": np.mean(y_true),
            "pred_mean": np.mean(y_pred),
            "true_std": true_std,
            "pred_std": pred_std,
            "std_ratio_pred_over_true": pred_std / true_std if true_std > 0 else np.nan,
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "R2": r2_score(y_true, y_pred)
        })

    return pd.DataFrame(rows)


spread_results = []

for test_file in TEST_FILES.keys():
    temp = variance_spread_diagnostic_from_saved_predictions(test_file)
    spread_results.append(temp)

if len(spread_results) > 0:
    spread_df = pd.concat(spread_results, axis=0, ignore_index=True)
    display(spread_df.sort_values("std_ratio_pred_over_true"))

    spread_path = os.path.join(OUTPUT_DIR, "variance_spread_diagnostic_4net_residual_ensemble_nn.csv")
    spread_df.to_csv(spread_path, index=False)
    print("Saved variance spread diagnostic to:", spread_path)


,test_file,output,n,true_mean,pred_mean,true_std,pred_std,std_ratio_pred_over_true,MAE,RMSE,R2
23,dat_test_qq_t.csv,var_bc_2,25,-7.174606,-7.315045,2.432747,0.008952,0.003680,1.750159,2.438562,-0.004786
22,dat_test_qq_t.csv,var_bc_1,25,-6.201920,-6.612600,2.053812,0.009208,0.004484,1.627437,2.094194,-0.039711
21,dat_test_qq_t.csv,var_log_max_strain,25,-12.441455,-6.999730,18.338812,0.166882,0.009100,7.909116,19.136488,-0.088885
8,dat_test_bin_even_t.csv,var_bc_1,39,-6.493181,-6.598667,1.727000,0.024327,0.014086,1.198973,1.728455,-0.001686
9,dat_test_bin_even_t.csv,var_bc_2,39,-7.299178,-7.286622,1.559687,0.037062,0.023762,1.150728,1.559969,-0.000361
24,dat_test_qq_t.csv,var_bc_3,25,-4.680151,-5.551217,1.900825,0.049084,0.025823,1.817301,2.076388,-0.193254
10,dat_test_bin_even_t.csv,var_bc_3,39,-4.875930,-5.503134,2.025438,0.086805,0.042858,1.611803,2.109535,-0.084765
25,dat_test_qq_t.csv,var_bc_4,25,-5.011874,-6.302551,2.219078,0.105167,0.047392,2.111553,2.609407,-0.382733
27,dat_test_qq_t.csv,var_bc_6,25,-3.093343,-4.456520,2.127405,0.110519,0.051950,1.957507,2.550451,-0.437255
19,dat_test_qq_c.csv,var_bc_5,25,-5.202517,-5.916215,2.163009,0.112910,0.052201,1.523864,2.269434,-0.100826


Saved variance spread diagnostic to: C:\Users\AbsoluteArm\OneDrive - University of Oklahoma (1)\Alloy Project\6coef_May25\pred\variance_spread_diagnostic_4net_residual_ensemble_nn.csv
